<a href="https://colab.research.google.com/github/ZeninKris/zmm-movilidad-predictiva/blob/main/notebooks/05_Feature_Engineering_OCISEVI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — Feature Engineering: Siniestros OCISEVI

## Objetivo
Transformar los 203,890 registros individuales de siniestros
(una fila por accidente) en features agregadas por hora,
para integrarlos al esqueleto temporal (una fila por hora).

## El problema central
El esqueleto tiene granularidad horaria.
Los siniestros tienen granularidad de evento individual.
Hay que "comprimir" los siniestros a nivel hora mediante groupby.

## Features construidas
- `siniestros_zona_industrial` — siniestros por hora en 7 municipios industriales
- `sin_apodaca/guadalupe/escobedo/garcia/juarez/pesqueria/santa_catarina` — conteo por municipio
- `Total de lesionados` / `Total de fallecidos` — gravedad agregada por hora
- `tiene_coordenadas` — cuántos siniestros de esa hora tienen ubicación verificable
- `nivel_lluvia` / `temperatura_c` — clima imputado con Open-Meteo (0 nulos)

## Municipios industriales — lista definitiva
7 municipios respaldados por DENUE (SCIAN 31-33/48-49, >30 empleados):
Apodaca (533), Guadalupe (280), Santa Catarina (244), Escobedo (208),
García (80), Pesquería (45), Juárez (31)

## Decisiones clave
- Granularidad horaria: consistente con el esqueleto temporal
- Guadalupe incluido: 280 empresas industriales, omitido en versión anterior
- NaN → 0 tras el merge: horas sin accidentes son información real, no datos faltantes
- Tipo vehículo y Tipo vialidad conservados: señal real para el modelo

In [1]:
# ============================================================
# CELDA 1 — Carga de datos
# ============================================================

import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

RAW       = '/content/drive/MyDrive/Proyecto_ZMM/data_raw/'
PROCESSED = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/'

# --- 1.1 Cargar siniestros unificados (output del Notebook 04) ---
df = pd.read_csv(PROCESSED + 'rativ_unificado.csv', low_memory=False)
print(f"Siniestros cargados: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# --- 1.2 Cargar esqueleto temporal con clima ---
esqueleto = pd.read_csv(PROCESSED + 'esqueleto_tiempo_eventos_clima.csv')
esqueleto['fecha_hora'] = pd.to_datetime(esqueleto['fecha_hora'])
print(f"Esqueleto cargado:   {esqueleto.shape[0]:,} filas × {esqueleto.shape[1]} columnas")

# --- 1.3 Vista rápida ---
print(f"\nColumnas siniestros: {df.columns.tolist()}")
print(f"\nPrimeras filas:")
print(df.head(3))

Mounted at /content/drive
Siniestros cargados: 203,890 filas × 17 columnas
Esqueleto cargado:   35,064 filas × 13 columnas

Columnas siniestros: ['MUNICIPIO', 'Día', 'Mes', 'Año', 'Día de la semana', 'Hora de reporte', 'Calle', 'Colonia', 'Referencia', 'Tipo de Hecho de Tránsito', 'Condición de clima', 'VEHICULO CODIFICADO', 'Causa del HT', 'Total de lesionados', 'Total de fallecidos', 'anio_fuente', 'Tipo de calle']

Primeras filas:
   MUNICIPIO  Día  Mes   Año  Día de la semana Hora de reporte  \
0         31    1    1  2023                 1        02:41:00   
1         31    1    1  2023                 1        06:30:00   
2         31    1    1  2023                 1        07:20:00   

                  Calle Colonia Referencia Tipo de Hecho de Tránsito  \
0  ALEJANDRO GARZA LEAL      99         99         Choque de crucero   
1  ARTURO B DE LA GARZA      99         99            Estrellamiento   
2                JAZMIN      99         99            Estrellamiento   

   Condi

In [2]:
# ============================================================
# CELDA 2 — Construir fecha_hora por siniestro
# ============================================================

# --- 2.1 Parsear hora de reporte ---
# Viene como string 'HH:MM:SS' — extraemos solo la hora (granularidad horaria)
df['hora'] = pd.to_datetime(
    df['Hora de reporte'], format='%H:%M:%S', errors='coerce'
).dt.hour

# --- 2.2 Construir fecha_hora como timestamp horario ---
# Combinamos Año + Mes + Día + hora para obtener el índice temporal
df['fecha_hora'] = pd.to_datetime({
    'year':  df['Año'],
    'month': df['Mes'],
    'day':   df['Día'],
    'hour':  df['hora']
})

# --- 2.3 Diagnóstico ---
nulos_hora = df['hora'].isna().sum()
nulos_fh   = df['fecha_hora'].isna().sum()

print(f"Hora de reporte no parseada: {nulos_hora:,} ({nulos_hora/len(df)*100:.1f}%)")
print(f"fecha_hora nula:             {nulos_fh:,}  ({nulos_fh/len(df)*100:.1f}%)")

print(f"\nRango temporal de siniestros:")
print(f"  Desde: {df['fecha_hora'].min()}")
print(f"  Hasta: {df['fecha_hora'].max()}")

print(f"\nEjemplos:")
print(df[['Año','Mes','Día','Hora de reporte','hora','fecha_hora']].head(5))

Hora de reporte no parseada: 40,779 (20.0%)
fecha_hora nula:             40,779  (20.0%)

Rango temporal de siniestros:
  Desde: 2023-01-01 00:00:00
  Hasta: 2025-12-31 23:00:00

Ejemplos:
    Año  Mes  Día Hora de reporte  hora          fecha_hora
0  2023    1    1        02:41:00   2.0 2023-01-01 02:00:00
1  2023    1    1        06:30:00   6.0 2023-01-01 06:00:00
2  2023    1    1        07:20:00   7.0 2023-01-01 07:00:00
3  2023    1    1        08:32:00   8.0 2023-01-01 08:00:00
4  2023    1    1        11:07:00  11.0 2023-01-01 11:00:00


In [3]:
# ============================================================
# CELDA 3 — Diagnóstico: ¿por qué falla el 20% de horas?
# ============================================================

# --- 3.1 Ver los valores que no pudieron parsearse ---
mask_nulos = df['hora'].isna()
valores_fallidos = df.loc[mask_nulos, 'Hora de reporte'].value_counts().head(20)

print(f"Top 20 valores que no parsearon:")
print(valores_fallidos)

# --- 3.2 ¿Se concentran en algún año? ---
print(f"\nDistribución de fallas por año:")
print(df.loc[mask_nulos, 'anio_fuente'].value_counts().sort_index())

# --- 3.3 ¿Cuántos son código 99? ---
es_99 = df.loc[mask_nulos, 'Hora de reporte'].isin(['99', 99, '99:99:99'])
print(f"\nDe los {mask_nulos.sum():,} fallidos:")
print(f"  Son código 99:     {es_99.sum():,}")
print(f"  Otro formato:      {(~es_99).sum():,}")

Top 20 valores que no parsearon:
Hora de reporte
SD       3291
99        819
14:00     152
17:00     150
19:00     149
09:00     143
08:00     138
15:00     136
16:00     127
15:20     127
18:00     122
13:00     119
20:00     117
15:30     114
08:10     110
08:30     109
10:00     105
14:50     102
16:40     101
14:20     101
Name: count, dtype: int64

Distribución de fallas por año:
anio_fuente
2023    16185
2024    14249
2025    10345
Name: count, dtype: int64

De los 40,779 fallidos:
  Son código 99:     819
  Otro formato:      39,960


In [4]:
# ============================================================
# CELDA 4 — Corregir parseo de hora: recuperar formato HH:MM
# ============================================================

# --- 4.1 Función que intenta los dos formatos ---
def parsear_hora(valor):
    """Intenta HH:MM:SS primero, luego HH:MM. Devuelve la hora (int) o NaN."""
    if pd.isna(valor) or str(valor).strip() in ['99', 'SD', 'NA']:
        return np.nan
    for fmt in ['%H:%M:%S', '%H:%M']:
        try:
            return pd.to_datetime(str(valor).strip(), format=fmt).hour
        except ValueError:
            continue
    return np.nan

# --- 4.2 Re-parsear la columna hora con la función corregida ---
df['hora'] = df['Hora de reporte'].apply(parsear_hora)

# --- 4.3 Reconstruir fecha_hora ---
df['fecha_hora'] = pd.to_datetime({
    'year':  df['Año'],
    'month': df['Mes'],
    'day':   df['Día'],
    'hour':  df['hora']
}, errors='coerce')

# --- 4.4 Diagnóstico final ---
nulos = df['fecha_hora'].isna().sum()
print(f"Registros sin fecha_hora válida: {nulos:,} ({nulos/len(df)*100:.1f}%)")
print(f"Registros recuperados: {40779 - nulos:,}")
print(f"Rango temporal:")
print(f"  Desde: {df['fecha_hora'].min()}")
print(f"  Hasta: {df['fecha_hora'].max()}")

Registros sin fecha_hora válida: 4,327 (2.1%)
Registros recuperados: 36,452
Rango temporal:
  Desde: 2023-01-01 00:00:00
  Hasta: 2025-12-31 23:00:00


In [5]:
# ===========================================================
# CELDA 5 (Corregida) — Limpieza de nulos irrecuperables y flag espacial
# ===========================================================

# 1. Eliminar filas donde no pudimos rescatar la fecha_hora (los SD/99 temporales)
print(f"Total de registros ANTES de limpiar nulos temporales: {len(df)}")
df = df.dropna(subset=['fecha_hora']).copy()
print(f"Total de registros DESPUÉS: {len(df)}")

# 2. Crear flag 'tiene_coordenadas' basándonos en la columna 'Referencia'
# Es válido si no es nulo y no es el código '99' (ni en texto ni en número)
df['tiene_coordenadas'] = df['Referencia'].notna() & (df['Referencia'] != '99') & (df['Referencia'] != 99)

# 3. Ver cómo se distribuyen las coordenadas por año
print("\nDesglose de registros CON coordenadas por año:")
print(df.groupby('Año')['tiene_coordenadas'].value_counts().unstack().fillna(0))

Total de registros ANTES de limpiar nulos temporales: 203890
Total de registros DESPUÉS: 199563

Desglose de registros CON coordenadas por año:
tiene_coordenadas    False    True 
Año                                
2023               55829.0  15261.0
2024                   0.0  66681.0
2025               27107.0  34685.0


In [6]:
# ===========================================================
# CELDA 6 — Enriquecimiento y reemplazo de Clima con Open-Meteo
# ===========================================================
import pandas as pd

# 1. Cargar el dataset de clima oficial (Ajusta el nombre del archivo si es necesario)
ruta_clima = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos_clima.csv' # <-- REVISA ESTE NOMBRE
df_clima = pd.read_csv(ruta_clima)

# Asegurar formato datetime para que el cruce sea exacto
df_clima['fecha_hora'] = pd.to_datetime(df_clima['fecha_hora'])

# 2. Hacer el cruce (merge) por 'fecha_hora'
# Le pegamos a nuestros accidentes la temperatura y el nivel de lluvia de esa hora exacta
print(f"Columnas antes del cruce: {df.shape[1]}")
df = df.merge(df_clima[['fecha_hora', 'nivel_lluvia', 'temperatura_c']], on='fecha_hora', how='left')
print(f"Columnas después del cruce: {df.shape[1]}")

# 3. Verificamos que no haya nulos generados en el clima
print("\nNulos en datos de clima tras el cruce:")
print(df[['nivel_lluvia', 'temperatura_c']].isnull().sum())

# 4. (Opcional pero recomendado) Eliminar la columna original de Clima de OCISEVI
# porque ya tenemos nuestra "fuente de verdad" de Open-Meteo
if 'Clima' in df.columns:
    df = df.drop(columns=['Clima'])
    print("\nColumna original 'Clima' eliminada por redundancia/imprecisión.")

Columnas antes del cruce: 20
Columnas después del cruce: 22

Nulos en datos de clima tras el cruce:
nivel_lluvia     0
temperatura_c    0
dtype: int64


In [7]:
# ===========================================================
# CELDA 7 (Triunfal) — Seleccionar y limpiar columnas relevantes
# ===========================================================

print("Columnas antes de limpiar:")
print(df.columns.tolist())

# Nuestra lista VIP, ahora con tu variable rescatada
columnas_utiles = [
    'fecha_hora',             # Nuestra llave maestra
    'MUNICIPIO',              # MAYÚSCULAS: Para saber si fue en zona industrial
    'Total de lesionados',    # Para saber si fue un choque grave
    'Total de fallecidos',    # Para saber si fue un choque grave
    'VEHICULO CODIFICADO',    # Transporte pesado (códigos 3 y 4)
    'Tipo de calle',          # ¡La columna que rescataste con éxito!
    'tiene_coordenadas',      # Nuestra flag espacial
    'nivel_lluvia',           # El clima oficial
    'temperatura_c'           # Temperatura de Open-Meteo
]

# Sobrescribimos df solo con estas columnas
df = df[columnas_utiles].copy()

print(f"\nNos quedamos con {df.shape[1]} columnas clave para el modelo:")
print(df.columns.tolist())
print(df['MUNICIPIO'].value_counts().head(20))

Columnas antes de limpiar:
['MUNICIPIO', 'Día', 'Mes', 'Año', 'Día de la semana', 'Hora de reporte', 'Calle', 'Colonia', 'Referencia', 'Tipo de Hecho de Tránsito', 'Condición de clima', 'VEHICULO CODIFICADO', 'Causa del HT', 'Total de lesionados', 'Total de fallecidos', 'anio_fuente', 'Tipo de calle', 'hora', 'fecha_hora', 'tiene_coordenadas', 'nivel_lluvia', 'temperatura_c']

Nos quedamos con 9 columnas clave para el modelo:
['fecha_hora', 'MUNICIPIO', 'Total de lesionados', 'Total de fallecidos', 'VEHICULO CODIFICADO', 'Tipo de calle', 'tiene_coordenadas', 'nivel_lluvia', 'temperatura_c']
MUNICIPIO
39    96542
46    28612
19    19224
6     11311
21    10992
48     8852
26     8317
18     6267
31     3576
49     3300
41     2570
Name: count, dtype: int64


In [8]:
import pandas as pd

denue = pd.read_csv('/content/drive/MyDrive/Proyecto_ZMM/data_processed/denue_industrial_zmm_limpio.csv')

print("\n=== Columnas disponibles ===")
print(denue.columns.tolist())


=== Columnas disponibles ===
['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'entidad', 'cve_mun', 'municipio', 'cve_loc', 'localidad', 'ageb', 'manzana', 'telefono', 'correoelec', 'www', 'tipoUniEco', 'latitud', 'longitud', 'fecha_alta']


In [9]:
print(denue['municipio'].value_counts())

municipio
Apodaca                     533
Monterrey                   452
Guadalupe                   280
Santa Catarina              244
General Escobedo            208
San Nicolás de los Garza    204
García                       80
San Pedro Garza García       48
Salinas Victoria             45
Pesquería                    45
Juárez                       31
Santiago                     12
Hidalgo                       5
Name: count, dtype: int64


In [10]:
# Ver sectores SCIAN presentes y rangos de empleados
denue['sector_scian'] = denue['codigo_act'].astype(str).str[:2]

print("=== Distribución por sector SCIAN ===")
print(denue['sector_scian'].value_counts().head(20))

print("\n=== Valores únicos de per_ocu ===")
print(denue['per_ocu'].value_counts())

=== Distribución por sector SCIAN ===
sector_scian
33    902
32    587
48    451
31    198
49     49
Name: count, dtype: int64

=== Valores únicos de per_ocu ===
per_ocu
251 y más personas    574
51 a 100 personas     561
101 a 250 personas    537
31 a 50 personas      515
Name: count, dtype: int64


In [23]:
# ============================================================
# CELDA 8 (CORREGIDA) — Agregación con San Nicolás
# ============================================================

# 1. Mapeo actualizado (Incluye San Nicolás ID: 46)
mapa_municipios = {
    6:  'apodaca',
    19: 'guadalupe',
    18: 'escobedo',
    21: 'garcia',
    26: 'juarez',
    41: 'pesqueria',
    49: 'santa_catarina',
    46: 'san_nicolas'  # <--- Agregado aquí
}

municipios_industriales = list(mapa_municipios.keys())

# 2. Generar flags por municipio
for codigo, nombre in mapa_municipios.items():
    df[f'sin_{nombre}'] = (df['MUNICIPIO'] == codigo).astype(int)

# El flag industrial ahora incluye automáticamente a San Nicolás
df['flag_industrial'] = df['MUNICIPIO'].isin(municipios_industriales).astype(int)

# 3. Diccionario de agregación dinámico
columnas_agg = {
    'flag_industrial':     'sum',
    'Total de lesionados': 'sum',
    'Total de fallecidos': 'sum',
    'tiene_coordenadas':   'sum'
}

# Añadir automáticamente todos los sin_municipio al groupby
for nombre in mapa_municipios.values():
    columnas_agg[f'sin_{nombre}'] = 'sum'

df_por_hora = df.groupby('fecha_hora').agg(columnas_agg).reset_index()

df_por_hora = df_por_hora.rename(columns={
    'flag_industrial': 'siniestros_zona_industrial'
})



In [24]:
# ============================================================
# CELDA 9 (CORREGIDA) — Merge Final Limpio
# ============================================================

super_tabla = esqueleto.merge(df_por_hora, on='fecha_hora', how='left')

# Llenar ceros en todas las columnas de siniestros detectadas
cols_siniestros = [c for c in super_tabla.columns if c.startswith('sin_') or
                   c in ['siniestros_zona_industrial', 'Total de lesionados',
                         'Total de fallecidos', 'tiene_coordenadas']]

super_tabla[cols_siniestros] = super_tabla[cols_siniestros].fillna(0).astype(int)

# Guardado final
super_tabla.to_csv(PROCESSED + 'super_tabla_completa.csv', index=False)
print(f"✅ Super Tabla lista con {super_tabla.shape[1]} columnas y San Nicolás incluido.")

✅ Super Tabla lista con 25 columnas y San Nicolás incluido.


In [25]:
# ============================================================
# CELDA 10 — Verificación final y guardado
# ============================================================

print("=== Resumen de la Super Tabla ===")
print(f"Filas:    {super_tabla.shape[0]:,}")
print(f"Columnas: {super_tabla.shape[1]}")

print("\n=== Columnas completas ===")
print(super_tabla.columns.tolist())

print("\n=== Nulos por columna ===")
nulos = super_tabla.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✓ Sin nulos")

print("\n=== Tipos de dato ===")
print(super_tabla.dtypes)

print("\n=== Estadísticas clave ===")
cols_check = [
    'siniestros_zona_industrial', 'sin_apodaca', 'sin_guadalupe',
    'Total de lesionados', 'Total de fallecidos'
]
print(super_tabla[cols_check].describe())

# --- Guardado ---
ruta_output = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/super_tabla_completa.csv'
super_tabla.to_csv(ruta_output, index=False)
print(f"\n✓ Super Tabla guardada en: {ruta_output}")

=== Resumen de la Super Tabla ===
Filas:    35,064
Columnas: 25

=== Columnas completas ===
['fecha_hora', 'año', 'mes', 'dia_semana', 'hora_del_dia', 'es_fin_de_semana', 'intensidad_hora_pico', 'tipo_dia', 'asistencia_estimada', 'nivel_impacto', 'impacto_evento_activo', 'temperatura_c', 'nivel_lluvia', 'siniestros_zona_industrial', 'Total de lesionados', 'Total de fallecidos', 'tiene_coordenadas', 'sin_apodaca', 'sin_guadalupe', 'sin_escobedo', 'sin_garcia', 'sin_juarez', 'sin_pesqueria', 'sin_santa_catarina', 'sin_san_nicolas']

=== Nulos por columna ===
✓ Sin nulos

=== Tipos de dato ===
fecha_hora                    datetime64[ns]
año                                    int64
mes                                    int64
dia_semana                             int64
hora_del_dia                           int64
es_fin_de_semana                       int64
intensidad_hora_pico                   int64
tipo_dia                              object
asistencia_estimada                  float

# Cierre NB05 — Estado final

## Output generado
`super_tabla_completa.csv` — 35,064 filas × 25 columnas — 0 nulos

## Columnas de la Super Tabla
| Grupo | Columnas |
|-------|----------|
| Temporal | fecha_hora, año, mes, dia_semana, hora_del_dia, es_fin_de_semana, intensidad_hora_pico, tipo_dia |
| Eventos | asistencia_estimada, nivel_impacto, impacto_evento_activo |
| Clima | temperatura_c, nivel_lluvia |
| Siniestros | siniestros_zona_industrial, sin_apodaca, sin_guadalupe, sin_escobedo, sin_garcia, sin_juarez, sin_pesqueria, sin_santa_catarina, sin_san_nicolas Total de lesionados, Total de fallecidos, tiene_coordenadas |

## Siguiente paso — NB06
Variable espacial fina por parque industrial.
Requiere: CSV de Adrián con nombre, municipio, latitud, longitud de cada parque.
Método: distancia Haversine + buffer de 2km desde cada parque.